# YOLOv8 Image Classification

## Step 1: Install and check GPU

In [13]:
!pip install -q ultralytics

import os, torch
os.environ["WANDB_DISABLED"] = "true"
from ultralytics import YOLO
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True Tesla T4


## Step 2: Check the dataset layout

Expected: `seg_train/seg_train/<class>`, `seg_test/seg_test/<class>`, `seg_pred/seg_pred` (unlabeled).
If your output differs, adjust the paths in Step 3.

In [14]:
!find /kaggle/input -maxdepth 4 -type d | head -30

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/puneet6060
/kaggle/input/datasets/puneet6060/intel-image-classification
/kaggle/input/datasets/puneet6060/intel-image-classification/seg_pred
/kaggle/input/datasets/puneet6060/intel-image-classification/seg_test
/kaggle/input/datasets/puneet6060/intel-image-classification/seg_train
/kaggle/input/notebooks
/kaggle/input/notebooks/vincee


## Step 3: Link into the structure YOLO expects

YOLO classification needs a root with `train/` and `val/`. `seg_test` is used as `val`.

In [15]:
import os
from pathlib import Path

ROOT = Path("/kaggle/working/dataset")

# clean up any broken symlinks from the earlier attempt
for s in ["train", "val"]:
    p = ROOT / s
    if p.is_symlink() or p.exists():
        p.unlink()
ROOT.mkdir(exist_ok=True)

# find every folder that directly contains the class folders
found = []
for dirpath, dirnames, _ in os.walk("/kaggle/input"):
    if {"buildings", "forest", "glacier"}.issubset(set(dirnames)):
        found.append(Path(dirpath))

print("class folders found:")
for f in found:
    print(" ", f)

train_dir = next((f for f in found if "seg_train" in f.parts), None)
val_dir = next((f for f in found if "seg_test" in f.parts), None)

assert train_dir and val_dir, "Could not find train/test folders, paste the output above and I'll fix it"

os.symlink(train_dir, ROOT / "train")
os.symlink(val_dir, ROOT / "val")

for s in ["train", "val"]:
    classes = sorted(p.name for p in (ROOT / s).iterdir())
    count = len(list((ROOT / s).rglob("*.jpg")))
    print(s, count, classes)

class folders found:
  /kaggle/input/datasets/puneet6060/intel-image-classification/seg_test/seg_test
  /kaggle/input/datasets/puneet6060/intel-image-classification/seg_train/seg_train
train 14034 ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
val 3000 ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


## Step 4: Train

Roughly 10 to 15 minutes on a T4. Expect top-1 accuracy around 90 to 93%.

In [ ]:
model = YOLO("yolov8n-cls.pt")

results = model.train(
    data="/kaggle/working/dataset",
    epochs=30,
    imgsz=224,
    batch=64,
    device=0,
    patience=8,
    project="/kaggle/working/runs",
    name="intel_v1",
    plots=True,
)

Ultralytics 8.4.189 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=intel_v1, nbs=64, nms=None, opset=None, optimize=False, optim

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/30     0.902G     0.3065         18        224: 100% ━━━━━━━━━━━━ 220/220 4.7it/s 47.2s0.4ss
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 24/24 4.8it/s 5.0s0.2s
                   all      0.904          1

      Epoch    GPU_mem       loss  Instances       Size
       3/30     0.902G     0.2961         18        224: 100% ━━━━━━━━━━━━ 220/220 4.6it/s 48.0s0.4ss
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 24/24 4.8it/s 5.0s0.2s
                   all      0.914          1

      Epoch    GPU_mem       loss  Instances       Size
       4/30     0.902G     0.2825         18        224: 100% ━━━━━━━━━━━━ 220/220 4.6it/s 47.9s0.5ss
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 24/24 4.8it/s 5.0s0.2s
                   all      0.911          1

      Epoch    GPU_mem       loss  Instances       Size
       5/30     0.902G     0.2593         18        224: 100% ━━━━━━━━━━━━ 220/220 4.5it/s 48.7s0.5ss
               classes   

## Step 5: Evaluate

Top-5 is meaningless with 6 classes, so look at top-1. Check `confusion_matrix.png` and `results.png` in `/kaggle/working/runs/intel_v1/`.

In [ ]:
best = YOLO("/kaggle/working/runs/intel_v1/weights/best.pt")
m = best.val(data="/kaggle/working/dataset")
print("top1:", m.top1, "top5:", m.top5)

## Step 6: Test on unseen images (by eye, no labels in seg_pred)

In [ ]:
import random
from IPython.display import display
from PIL import Image

pred_dir = next(Path("/kaggle/input").rglob("seg_pred"))
pred_dir = pred_dir / "seg_pred" if (pred_dir / "seg_pred").exists() else pred_dir
samples = random.sample(list(pred_dir.glob("*.jpg")), 6)

for p in samples:
    r = best(p, verbose=False)[0]
    print(p.name, "->", r.names[r.probs.top1], round(float(r.probs.top1conf), 3))
    display(Image.open(p).resize((200, 200)))

## Step 7: Package the model

After this, use Save Version > Save & Run All, then download from the Output tab.
Or download straight from the file panel while the session is live.

In [ ]:
import shutil

best.export(format="onnx")  # optional, lighter for deployment
shutil.make_archive("/kaggle/working/intel_v1_export", "zip",
                    "/kaggle/working/runs/intel_v1/weights")
print(os.listdir("/kaggle/working"))